# ZINC Michael acceptors

Notebook to collect ZINC data and load it into tranches (n.b. this could take quite a long time; around a few hours to 1 day, depending on parallelization core count)

## Import necessary packages

In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import PandasTools
from tqdm import tqdm
import io, os
import requests
from huggingface_hub import hf_hub_url
from pandarallel import pandarallel

## Function to load ZINC datasets from Huggingface

In [2]:

def get_ZINC(chunk="00"):
    # Download, unzip and load ZINC data from zpn/zinc20 (approx. 10M molecules)
    # Choose the file to download
    repo_id="zpn/zinc20"
    filename = f"zinc_processed/smiles_all_{chunk}_clean.jsonl.gz"
    # Get the direct download URL from Hugging Face
    url = hf_hub_url(repo_id=repo_id, filename=filename, repo_type="dataset")
    # print(url)
    # Stream the file directly into memory
    response = requests.get(url)
    response.raise_for_status()
    df = pd.read_json(io.BytesIO(response.content), compression="gzip", lines=True)
    return df

## Functions to filter ZINC for Michael acceptors and remove unwanted substructures

In [3]:
smarts = "O=CC=C"
patt = Chem.MolFromSmarts(smarts)

filter_smarts = ["O=C([OH])C=C","O=C([O-])C=C",
                 "O=C(F)","O=C(Cl)","O=C(Br)","O=C(I)",
                 "O=CC([OH])=C","O=CC=C[OH]",
                 "O=CC([NH2])=C","O=CC=C[NH2]",
                 "O=CC=CC=O"]
filters = [Chem.MolFromSmarts(fs) for fs in filter_smarts]

def MatchMichael(df, patt=patt, ncpu=10):
    # (Parallel) match Michael acceptor substructure (~ 2 mins)
    pandarallel.initialize(nb_workers=ncpu, progress_bar=False, verbose=0)
    df.loc[:, "HasMatch"] = df["smiles"].parallel_apply(lambda x: Chem.MolFromSmiles(x).HasSubstructMatch(patt))
    filtered_df = df[df["HasMatch"]==True]
    filtered_df = filtered_df.drop("HasMatch", axis=1)
    return filtered_df

def FilterSubstruct(df, filters=filters):
    # Filter unwanted (reactive) substructures (~ 1 min)
    PandasTools.AddMoleculeColumnToFrame(df, smilesCol='smiles')
    df.loc[:, "Filter"] = df["ROMol"].apply(lambda x: any(x.HasSubstructMatch(patt) for patt in filters))
    filtered_df = df[df["Filter"]==False]
    filtered_df = filtered_df.drop(["ROMol","Filter"], axis=1)
    return filtered_df

## Run full-pass through ALL of ZINC-20 (approx. 1B molecules)

In [ ]:
# Choose a subset of 1B ZINC20 molecules to load
# each subset is 1/100th, so approx. 10M molecules

ncpu = 10 # we currently use 10 CPU cores for parallelization
chunks = [f'{n:02}' for n in range(100)]
for chunk in tqdm(chunks[:]): # if this crashes, continue using list slices, e.g. chunks[50:]
    df = MatchMichael(get_ZINC(chunk=chunk), ncpu=ncpu)
    fdf = FilterSubstruct(df)
    outfile = os.path.join("data","ZINC","tranches",f"ZINC_{chunk}_Michael.csv.gz")
    fdf.to_csv(outfile,index=False,compression='gzip')
